# CSL7110 — Assignment 4
## Part 1: Clustering | Part 2: Web Search (Inverted Index) | Part 3: PageRank on Spark

**Instructions recap:**
- Part 1: Farthest-First (k-center) + K-Means++ on the UCI Spambase dataset
- Part 2: Build an Inverted Index search engine and simulate actions from `actions.txt`
- Part 3: Implement PageRank on Spark using the provided graph dataset

---
## Setup — Install Dependencies

In [1]:
# Install PySpark (required for Parts 1 and 3)
!pip install pyspark -q

# Standard imports used throughout
import math
import random
import time
import re
import os
import numpy as np

print('All imports successful.')

All imports successful.


---
# PART 1: Clustering
### Dataset: UCI Spambase (4601 points, 57 features — last column is label, excluded from clustering)

## 1.1 — Initialize Spark

In [2]:
from pyspark import SparkContext, SparkConf
from pyspark.mllib.linalg import Vectors

conf = SparkConf().setAppName('Assignment4_Clustering').setMaster('local[*]')
sc = SparkContext(conf=conf)
sc.setLogLevel('ERROR')
print('Spark initialized:', sc.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/13 07:43:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark initialized: 4.0.2


## 1.2 — `readVectorsSeq`: Load dataset as list of Vectors
Each line is a comma-separated row. The last column is the class label (spam/not-spam)  
and is **excluded** from the feature vectors used for clustering.

In [ ]:
def readVectorsSeq(filename):
    """
    Reads a CSV file where each row is a feature point.
    The last column is the class label and is excluded.
    Returns a list of pyspark.mllib.linalg.Vector objects.
    """
    points = []
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            values = [float(x) for x in line.split(',')]
            # Exclude the last column (class label)
            points.append(Vectors.dense(values[:-1]))
    return points

# Upload spambase.data to Kaggle and update path accordingly
SPAMBASE_PATH = '/kaggle/input/datasets/akankshakapil/assignment4-mlbd/Assignment 4- datasets/Q1- UCI Spam clustering/spambase.data'  # Update this path if needed on Kaggle
P = readVectorsSeq(SPAMBASE_PATH)
print(f'Loaded {len(P)} points with {len(P[0])} dimensions each.')

In [8]:
import numpy as np

def sqdist(v1, v2):
    """Squared Euclidean distance between two pyspark Vectors."""
    diff = v1.toArray() - v2.toArray()
    return float(np.dot(diff, diff))

## 1.3 — `kcenter`: Farthest-First Traversal (k-Center Clustering)

**Algorithm:**
1. Pick the first center arbitrarily (index 0).
2. For each remaining step, pick the point farthest from all current centers.
3. Repeat until k centers are chosen.

**Time complexity:** O(|P| × k) — each of the k steps scans all points once.

In [9]:
def kcenter(P, k):
    """
    Farthest-First Traversal algorithm for k-center clustering.
    
    Args:
        P: list of pyspark.mllib.linalg.Vector
        k: number of centers
    Returns:
        C: list of k center Vectors
    """
    # Assumption: first center is the first point in P
    centers = [P[0]]
    
    # min_dist[i] = squared distance from P[i] to its closest center so far
    min_dist = [sqdist(p, centers[0]) for p in P]

    
    for _ in range(k - 1):
        # Pick the point with the maximum distance to the nearest center
        farthest_idx = max(range(len(P)), key=lambda i: min_dist[i])
        new_center = P[farthest_idx]
        centers.append(new_center)
        
        # Update min distances with the new center
        for i in range(len(P)):
            d = sqdist(P[i], new_center)
            if d < min_dist[i]:
                min_dist[i] = d
    
    return centers

print('kcenter function defined.')

kcenter function defined.


## 1.4 — `kmeansPP`: K-Means++ Seeding

**Algorithm:**
1. Pick the first center uniformly at random.
2. For each remaining step, pick the next center with probability proportional to its squared distance to the nearest existing center.
3. Repeat until k centers are chosen.

**Time complexity:** O(|P| × k)

In [10]:
def kmeansPP(P, k):
    """
    K-Means++ seeding algorithm.
    
    Args:
        P: list of pyspark.mllib.linalg.Vector
        k: number of centers
    Returns:
        C: list of k center Vectors chosen by k-means++ seeding
    """
    # Step 1: Pick first center uniformly at random
    centers = [random.choice(P)]
    
    # min_dist[i] = squared distance from P[i] to its closest center so far
    min_dist = [sqdist(p, centers[0]) for p in P]
    
    for _ in range(k - 1):
        total = sum(min_dist)
        
        # Sample next center with probability proportional to squared distance
        r = random.uniform(0, total)
        cumulative = 0.0
        chosen_idx = 0
        for i, d in enumerate(min_dist):
            cumulative += d
            if cumulative >= r:
                chosen_idx = i
                break
        
        new_center = P[chosen_idx]
        centers.append(new_center)
        
        # Update min distances
        for i in range(len(P)):
            d = sqdist(P[i], new_center)
            if d < min_dist[i]:
                min_dist[i] = d
    
    return centers

print('kmeansPP function defined.')

kmeansPP function defined.


## 1.5 — `kmeansObj`: K-Means Objective Function
Returns the **average squared distance** of each point in P to its nearest center in C.

In [11]:
def kmeansObj(P, C):
    """
    Computes the k-means objective: average squared distance from each
    point in P to its nearest center in C.
    
    Args:
        P: list of Vector (data points)
        C: list of Vector (cluster centers)
    Returns:
        float: average squared distance
    """
    total = 0.0
    for p in P:
        # Distance to nearest center
        min_d = min(sqdist(p, c) for c in C)
        total += min_d
    return total / len(P)

print('kmeansObj function defined.')

kmeansObj function defined.


## 1.6 — Main Program: Run all 3 experiments

**Assumptions:**
- k and k1 are set to 10 and 30 respectively as example values. Adjust as needed.
- The last column of spambase.data is the class label and is excluded from feature vectors.
- For kcenter, the first center is always P[0] (deterministic).

In [13]:
# Configuration 
k  = 10   # Number of final centers
k1 = 200   # Larger coreset size for Experiment 3 (k1 > k)

random.seed(42)  # For reproducibility


# Experiment 1: kcenter(P, k) — measure running time

print('=' * 60)
print(f'Experiment 1: kcenter(P, k={k})')
start = time.time()
C_kcenter = kcenter(P, k)
elapsed = time.time() - start
print(f'  Running time: {elapsed:.4f} seconds')
print(f'  Number of centers returned: {len(C_kcenter)}')


# Experiment 2: kmeansPP(P, k) then kmeansObj(P, C)

print('=' * 60)
print(f'Experiment 2: kmeansPP(P, k={k}) + kmeansObj')
C_kmeanspp = kmeansPP(P, k)
obj2 = kmeansObj(P, C_kmeanspp)
print(f'  kmeansObj (kmeans++ centers): {obj2:.6f}')


# Experiment 3: kcenter(P, k1) → kmeansPP(X, k) → kmeansObj(P, C)
# Idea: use k1 > k kcenter points as a coreset, then run kmeans++ on it

print('=' * 60)
print(f'Experiment 3: kcenter(P, k1={k1}) → kmeansPP(X, k={k}) → kmeansObj')
X = kcenter(P, k1)          # Coreset of k1 points
C_combined = kmeansPP(X, k)  # k centers from the coreset
obj3 = kmeansObj(P, C_combined)
print(f'  kmeansObj (coreset + kmeans++ centers): {obj3:.6f}')
print('=' * 60)
print('\nObservation: Experiment 3 should give an objective value comparable')
print('to Experiment 2, since the k1-coreset provides good coverage before')
print('applying kmeans++. Larger k1 = better coreset = lower objective.')

Experiment 1: kcenter(P, k=10)
  Running time: 0.0956 seconds
  Number of centers returned: 10
Experiment 2: kmeansPP(P, k=10) + kmeansObj
  kmeansObj (kmeans++ centers): 31250.987038
Experiment 3: kcenter(P, k1=200) → kmeansPP(X, k=10) → kmeansObj
  kmeansObj (coreset + kmeans++ centers): 30937.852274

Observation: Experiment 3 should give an objective value comparable
to Experiment 2, since the k1-coreset provides good coverage before
applying kmeans++. Larger k1 = better coreset = lower objective.


---
# PART 2: Web Search — Inverted Index

We build an inverted index over a set of webpages and support:
- `addPage x` — add webpage `x` to the search engine
- `queryFindPagesWhichContainWord x` — find all pages containing word `x`
- `queryFindPositionsOfWordInAPage x y` — find positions of word `x` in page `y`

**Assumptions (per spec):**
- Words are lowercased.
- Stop words are **not stored** but positions are **still counted** when computing word indices.
- Punctuation is replaced with a space.
- Plural/singular treated as same (only the exact pairs listed in the spec).
- The connector words, punctuation, and plural mappings lists are treated as exhaustive.

## 2.1 — Text Preprocessing

In [14]:
# Stop words: not stored in index but positions still count
STOP_WORDS = {
    'a', 'an', 'the', 'they', 'these', 'this', 'for', 'is', 'are',
    'was', 'of', 'or', 'and', 'does', 'will', 'whose'
}

# Punctuation characters replaced by space
PUNCTUATION = set('{}[]<>=().,;\'"?#!-:')

# Plural → singular mapping (exhaustive as per spec)
# We normalize every word to its singular form before indexing
PLURAL_TO_SINGULAR = {
    'stacks': 'stack',
    'structures': 'structure',
    'applications': 'application',
}


def normalize_word(word):
    """
    Lowercase, apply plural→singular normalization.
    Returns normalized word string.
    """
    word = word.lower()
    return PLURAL_TO_SINGULAR.get(word, word)


def tokenize(text):
    """
    Replace punctuation with spaces, split into tokens,
    lowercase, and normalize plurals.
    Returns list of (position, normalized_word) tuples.
    Positions are 1-indexed and include stop-word positions.
    """
    # Replace each punctuation character with a space
    cleaned = ''.join(' ' if ch in PUNCTUATION else ch for ch in text)
    
    raw_tokens = cleaned.split()
    
    result = []  # (position, word) — position is 1-indexed
    for idx, token in enumerate(raw_tokens, start=1):
        word = normalize_word(token)
        result.append((idx, word))
    
    return result


print('Preprocessing utilities defined.')

Preprocessing utilities defined.


## 2.2 — Core Classes: Position, WordEntry, PageIndex, PageEntry

In [15]:
class MySet:
    """A simple set wrapper supporting union and intersection."""

    def __init__(self):
        self._data = set()

    def addElement(self, element):
        self._data.add(element)

    def union(self, otherSet):
        result = MySet()
        result._data = self._data | otherSet._data
        return result

    def intersection(self, otherSet):
        result = MySet()
        result._data = self._data & otherSet._data
        return result

    def __iter__(self):
        return iter(self._data)

    def __len__(self):
        return len(self._data)


class Position:
    """Represents a (PageEntry, word_position) tuple."""

    def __init__(self, page_entry, word_index):
        self._page_entry = page_entry
        self._word_index = word_index

    def getPageEntry(self):
        return self._page_entry

    def getWordIndex(self):
        return self._word_index


class WordEntry:
    """Stores all positions (across documents) where a specific word appears."""

    def __init__(self, word):
        self._word = word
        self._positions = []  # list of Position objects

    def addPosition(self, position):
        self._positions.append(position)

    def addPositions(self, positions):
        self._positions.extend(positions)

    def getAllPositionsForThisWord(self):
        return list(self._positions)

    def getTermFrequency(self, page_name, total_words_in_page):
        """
        Term frequency = (occurrences of word in page) / (total words in page).
        """
        count = sum(
            1 for pos in self._positions
            if pos.getPageEntry().getPageName() == page_name
        )
        if total_words_in_page == 0:
            return 0.0
        return count / total_words_in_page


class PageIndex:
    """
    Index for a single page: maps each unique (non-stop) word
    to its list of positions within that page.
    """

    def __init__(self):
        self._entries = {}  # word -> WordEntry

    def addPositionForWord(self, word, position):
        if word not in self._entries:
            self._entries[word] = WordEntry(word)
        self._entries[word].addPosition(position)

    def getWordEntries(self):
        return list(self._entries.values())

    def getWordEntry(self, word):
        return self._entries.get(word, None)

    def containsWord(self, word):
        return word in self._entries

    def getPositionsOfWord(self, word):
        entry = self._entries.get(word)
        if entry is None:
            return []
        return [pos.getWordIndex() for pos in entry.getAllPositionsForThisWord()]


class PageEntry:
    """
    Represents a webpage: reads the file, builds its PageIndex.
    """

    def __init__(self, page_name, webpages_folder='.'):
        self._page_name = page_name
        self._page_index = PageIndex()
        self._total_tokens = 0  # Total token count including stop words
        self._read_file(page_name, webpages_folder)

    def _read_file(self, page_name, folder):
        filepath = os.path.join(folder, page_name)
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            text = f.read()

        tokens = tokenize(text)  # list of (position, word)
        self._total_tokens = len(tokens)

        for (pos, word) in tokens:
            if word in STOP_WORDS:
                continue  # Don't index stop words; position still counted
            position = Position(self, pos)
            self._page_index.addPositionForWord(word, position)

    def getPageName(self):
        return self._page_name

    def getPageIndex(self):
        return self._page_index

    def getTotalTokens(self):
        return self._total_tokens


print('Core classes defined: MySet, Position, WordEntry, PageIndex, PageEntry')

Core classes defined: MySet, Position, WordEntry, PageIndex, PageEntry


## 2.3 — MyHashTable and InvertedPageIndex

In [16]:
class MyHashTable:
    """
    Hash table mapping a word to its WordEntry (global — across all pages).
    Uses Python's built-in dict as the underlying structure.
    """

    def __init__(self, size=1024):
        self._size = size
        self._table = {}  # word -> WordEntry

    def getHashIndex(self, word):
        """Hash function: sum of ASCII values mod table size."""
        return sum(ord(c) for c in word) % self._size

    def addPositionsForWord(self, word_entry):
        """
        Add or merge a WordEntry into the hash table.
        """
        word = word_entry._word
        if word not in self._table:
            self._table[word] = WordEntry(word)
        self._table[word].addPositions(word_entry.getAllPositionsForThisWord())

    def getWordEntry(self, word):
        return self._table.get(word, None)

    def containsWord(self, word):
        return word in self._table


class InvertedPageIndex:
    """
    Global inverted index over all added pages.
    Maps each word to the set of pages that contain it,
    along with each word's positions per page.
    """

    def __init__(self):
        self._hash_table = MyHashTable()
        self._pages = {}  # page_name -> PageEntry

    def addPage(self, page_entry):
        """Add a new page and update the global inverted index."""
        self._pages[page_entry.getPageName()] = page_entry
        for word_entry in page_entry.getPageIndex().getWordEntries():
            self._hash_table.addPositionsForWord(word_entry)

    def getPagesWhichContainWord(self, word):
        """
        Returns a MySet of PageEntry objects whose pages contain the word.
        """
        word = normalize_word(word)
        result = MySet()
        entry = self._hash_table.getWordEntry(word)
        if entry is None:
            return result
        seen = set()
        for pos in entry.getAllPositionsForThisWord():
            pe = pos.getPageEntry()
            if pe.getPageName() not in seen:
                result.addElement(pe)
                seen.add(pe.getPageName())
        return result

    def getPage(self, page_name):
        return self._pages.get(page_name, None)


print('MyHashTable and InvertedPageIndex defined.')

MyHashTable and InvertedPageIndex defined.


## 2.4 — SearchEngine and performAction

In [17]:
class SearchEngine:
    """
    Search engine backed by an InvertedPageIndex.
    Supports: addPage, queryFindPagesWhichContainWord,
              queryFindPositionsOfWordInAPage
    """

    def __init__(self, webpages_folder='.'):
        self._index = InvertedPageIndex()
        self._webpages_folder = webpages_folder

    def performAction(self, action_message):
        """
        Parse and execute a single action string.
        Returns the output string for that action.
        """
        parts = action_message.strip().split()
        if not parts:
            return ''

        action = parts[0]

        # addPage x 
        if action == 'addPage':
            page_name = parts[1]
            page_entry = PageEntry(page_name, self._webpages_folder)
            self._index.addPage(page_entry)
            return None  # addPage produces no output

        #  queryFindPagesWhichContainWord x 
        elif action == 'queryFindPagesWhichContainWord':
            word = normalize_word(parts[1])
            pages = self._index.getPagesWhichContainWord(word)
            if len(pages) == 0:
                return f'No webpage contains word {parts[1]}'
            # Sort for deterministic output
            names = sorted(pe.getPageName() for pe in pages)
            return ', '.join(names)

        #  queryFindPositionsOfWordInAPage x y 
        elif action == 'queryFindPositionsOfWordInAPage':
            word = normalize_word(parts[1])
            page_name = parts[2]
            page_entry = self._index.getPage(page_name)
            if page_entry is None:
                return f'No webpage {page_name} found'
            positions = page_entry.getPageIndex().getPositionsOfWord(word)
            if not positions:
                return f'Webpage {page_name} does not contain word {parts[1]}'
            return ', '.join(str(p) for p in sorted(positions))

        else:
            return f'Unknown action: {action}'


print('SearchEngine defined.')

SearchEngine defined.


## 2.5 — Run actions.txt and compare with answers.txt

> **Note for Kaggle:** Upload all webpage files and `actions.txt`/`answers.txt` to your Kaggle dataset.
> Update `WEBPAGES_FOLDER` to the correct path.

In [21]:
# Update these paths to match your Kaggle dataset upload location
WEBPAGES_FOLDER = '/kaggle/input/datasets/akankshakapil/assignment4-mlbd/Assignment 4- datasets/Q2- webSearch/webpages/'       # Folder containing the webpage files
ACTIONS_FILE    = '/kaggle/input/datasets/akankshakapil/assignment4-mlbd/Assignment 4- datasets/Q2- webSearch/actions.txt'
ANSWERS_FILE    = '/kaggle/input/datasets/akankshakapil/assignment4-mlbd/Assignment 4- datasets/Q2- webSearch/answers.txt'

engine = SearchEngine(webpages_folder=WEBPAGES_FOLDER)

# Read all actions
with open(ACTIONS_FILE, 'r') as f:
    actions = [line.strip() for line in f if line.strip()]

# Read expected answers (only query actions produce output)
with open(ANSWERS_FILE, 'r') as f:
    expected_answers = [line.strip() for line in f if line.strip()]

# Execute all actions, collect outputs
print('--- Running actions.txt ---\n')
actual_outputs = []
for action in actions:
    result = engine.performAction(action)
    if result is not None:  # addPage produces None
        actual_outputs.append(result)
        print(f'ACTION : {action}')
        print(f'OUTPUT : {result}')
        print()

# Compare with expected answers
print('\n--- Comparing with answers.txt ---\n')
all_correct = True
for i, (actual, expected) in enumerate(zip(actual_outputs, expected_answers)):
    match = (actual.strip() == expected.strip())
    status = '✓ PASS' if match else '✗ FAIL'
    if not match:
        all_correct = False
    print(f'[{status}] Expected: "{expected}"')
    if not match:
        print(f'         Got:      "{actual}"')

print()
if all_correct:
    print('All outputs match answers.txt!')
else:
    print('Some outputs differ from answers.txt — check above.')

--- Running actions.txt ---

ACTION : queryFindPagesWhichContainWord delhi
OUTPUT : No webpage contains word delhi

ACTION : queryFindPagesWhichContainWord stack
OUTPUT : stack_datastructure_wiki

ACTION : queryFindPagesWhichContainWord wikipedia
OUTPUT : stack_datastructure_wiki

ACTION : queryFindPositionsOfWordInAPage magazines stack_datastructure_wiki
OUTPUT : Webpage stack_datastructure_wiki does not contain word magazines

ACTION : queryFindPagesWhichContainWord allain
OUTPUT : No webpage contains word allain

ACTION : queryFindPagesWhichContainWord allain
OUTPUT : stack_cprogramming

ACTION : queryFindPagesWhichContainWord C
OUTPUT : stack_cprogramming

ACTION : queryFindPagesWhichContainWord C++
OUTPUT : stack_cprogramming

ACTION : queryFindPagesWhichContainWord jdk
OUTPUT : stack_oracle

ACTION : queryFindPagesWhichContainWord function
OUTPUT : stack_cprogramming, stack_datastructure_wiki, stackoverflow

ACTION : queryFindPagesWhichContainWord magazines
OUTPUT : stackmagazine

---
# PART 3: PageRank on Spark

**Dataset:** `small.txt` (53 nodes) and `whole.txt` (1000 nodes, 8192 edges)  
Download from: https://github.com/pnijhara/PySpark-PageRank/tree/main/graph

**PageRank formula:**  
r = ((1 - β) / n) × A + β × M × r  
where β = 0.8, A = unit vector, n = number of nodes

**Assumptions:**
- Multiple directed edges between the same pair of nodes are treated as a single edge.
- All nodes have positive out-degree (guaranteed by the dataset structure with cycle).
- We run 40 iterations as specified.

## 3.1 — Download Dataset

In [23]:
# Download the graph files from GitHub
# Files uploaded manually to Kaggle
# Update these paths to where you uploaded them in Kaggle

small_path = '/kaggle/input/datasets/akankshakapil/part3-assignment4/small (1).txt'
whole_path = '/kaggle/input/datasets/akankshakapil/part3-assignment4/whole.txt'

# Verify files loaded correctly
print('Verifying files...')
with open(small_path) as f:
    lines = f.readlines()
print(f'small.txt: {len(lines)} lines, first 5:')
for l in lines[:5]:
    print(' ', l.strip())

import subprocess
result = subprocess.run(['wc', '-l', whole_path], capture_output=True, text=True)
print('whole.txt:', result.stdout.strip())

Verifying files...
small.txt: 1024 lines, first 5:
  100	1
  13	1
  28	1
  89	1
  82	1
whole.txt: 8191 /kaggle/input/datasets/akankshakapil/part3-assignment4/whole.txt


## 3.2 — PageRank Implementation on Spark RDDs

In [24]:
def run_pagerank(filepath, beta=0.8, iterations=40):
    """
    Runs the iterative PageRank algorithm on Spark.

    Args:
        filepath : path to graph file (two space-separated columns: src dst)
        beta     : damping factor (default 0.8)
        iterations: number of iterations (default 40)

    Returns:
        sorted list of (node_id, pagerank_score) tuples
    """

    # Step 1: Load edges, deduplicate, build adjacency list
    # Each line: src dst (space separated)
    raw_edges = sc.textFile(filepath)

    # Parse edges and deduplicate (treat multiple edges as one)
    edges = (
        raw_edges
        .map(lambda line: tuple(int(x) for x in line.strip().split()))
        .distinct()                        # Remove duplicate directed edges
        .filter(lambda e: len(e) == 2)     # Safety: skip malformed lines
    )

    # Collect all unique node IDs
    all_nodes = (
        edges.flatMap(lambda e: [e[0], e[1]])
             .distinct()
             .collect()
    )
    n = len(all_nodes)
    print(f'  Graph loaded: {n} nodes')

    # Build adjacency list: src -> list of destinations
    adjacency = (
        edges
        .groupByKey()
        .mapValues(list)
    )

    # Out-degree of each source node
    out_degree = adjacency.mapValues(len)  # (src, out_deg)

    #  Step 2: Build M as (src, dst, weight) RDD
    # M[dst][src] = 1/deg(src) if edge src->dst exists
    # We represent M row contributions: for each edge (src, dst),
    # contribution = r[src] / deg(src) added to r[dst]

    # Join adjacency with out_degree to get (src, (neighbors, deg))
    # Then emit (dst, 1/deg) for each neighbor
    # We precompute weighted edges: (src, (dst, 1/deg(src)))
    weighted_edges = (
        adjacency
        .join(out_degree)              # (src, (neighbors_list, deg))
        .flatMap(lambda x: [
            (dst, (x[0], 1.0 / x[1][1]))   # (dst, (src, weight))
            for dst in x[1][0]
        ])
    )  # RDD of (dst, (src, weight))

    # Group by dst: (dst, list of (src, weight))
    # We restructure to (src, (dst, weight)) for easy lookup of r[src]
    # Better: keep as (dst, [(src, weight)])
    edges_by_src = (
        adjacency
        .join(out_degree)              # (src, ([dsts], deg))
        .flatMap(lambda x: [
            (x[0], dst, 1.0 / x[1][1])  # (src, dst, weight)
            for dst in x[1][0]
        ])
    )  # RDD of (src, dst, weight)

    # Step 3: Initialize PageRank vector r 
    # r is an RDD of (node_id, rank) pairs
    r = sc.parallelize(all_nodes).map(lambda v: (v, 1.0 / n))

    teleport = (1.0 - beta) / n  # Constant teleportation contribution

    # Step 4: Iterative update 
    for i in range(iterations):
        # For each edge (src, dst, weight), compute contribution = r[src] * weight
        # Then sum contributions by dst
        contributions = (
            edges_by_src
            .map(lambda e: (e[0], e[1], e[2]))  # (src, dst, weight)
            .keyBy(lambda e: e[0])               # (src, (src, dst, weight))
            .join(r)                             # (src, ((src, dst, weight), r_src))
            .map(lambda x: (x[1][0][1], x[1][0][2] * x[1][1]))  # (dst, contribution)
            .reduceByKey(lambda a, b: a + b)     # Sum contributions per dst
        )

        # New rank = teleport + beta * sum_contributions
        r_new = contributions.mapValues(lambda v: teleport + beta * v)

        # Handle dangling nodes: nodes that appear only as destinations
        # (their rank was not updated) — set to teleport value
        all_nodes_rdd = sc.parallelize([(v, teleport) for v in all_nodes])
        r = (
            all_nodes_rdd
            .leftOuterJoin(r_new)
            .mapValues(lambda v: v[1] if v[1] is not None else v[0])
        )
        r.cache()  # Cache for reuse in next iteration

    # Collect final PageRank vector
    final_ranks = sorted(r.collect(), key=lambda x: x[1], reverse=True)
    return final_ranks


print('run_pagerank function defined.')

run_pagerank function defined.


## 3.3 — Validate on small.txt (expected top score ≈ 0.036)

In [25]:
print('Running PageRank on small.txt (53 nodes)...')
small_ranks = run_pagerank(small_path, beta=0.8, iterations=40)

print(f'\nTop node score: {small_ranks[0][1]:.4f}  (expected ≈ 0.036)')
print('\nTop 5 nodes (small graph):')
for node, score in small_ranks[:5]:
    print(f'  Node {node:>4} : {score:.6f}')

Running PageRank on small.txt (53 nodes)...


  Graph loaded: 100 nodes



Top node score: 0.0357  (expected ≈ 0.036)

Top 5 nodes (small graph):
  Node   53 : 0.035731
  Node   14 : 0.034171
  Node   40 : 0.033630
  Node    1 : 0.030006
  Node   27 : 0.029720


## 3.4 — Run on whole.txt (1000 nodes, β=0.8, 40 iterations)

In [26]:
print('Running PageRank on whole.txt (1000 nodes, 40 iterations, β=0.8)...')
whole_ranks = run_pagerank(whole_path, beta=0.8, iterations=40)

print('\n--- Top 5 nodes (highest PageRank) ---')
for node, score in whole_ranks[:5]:
    print(f'  Node {node:>5} : {score:.8f}')

print('\n--- Bottom 5 nodes (lowest PageRank) ---')
for node, score in whole_ranks[-5:]:
    print(f'  Node {node:>5} : {score:.8f}')

Running PageRank on whole.txt (1000 nodes, 40 iterations, β=0.8)...
  Graph loaded: 1000 nodes



--- Top 5 nodes (highest PageRank) ---
  Node   263 : 0.00202029
  Node   537 : 0.00194334
  Node   965 : 0.00192545
  Node   243 : 0.00185263
  Node   285 : 0.00182737

--- Bottom 5 nodes (lowest PageRank) ---
  Node   408 : 0.00038780
  Node   424 : 0.00035482
  Node    62 : 0.00035315
  Node    93 : 0.00035136
  Node   558 : 0.00032860


In [ ]:
# Stop Spark session when done
sc.stop()
print('Spark session stopped.')